#Responsible Credit Scoring
##Predictive Risk Modeling & Ethical Fairness Auditing
###Student: Carlos Elizondo
###Student Number: 500890062
### Supervisor: Tamer Abdou
**Toronto Metropolitan University**

### Stage 1 & 2: Feature Selection & Relational Aggregation

In [1]:
# Installing libraries
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# PREVIOUS COMMANDS
import sys
!pip install ydata-profiling
!jupyter nbextension enable --py widgetsnbextension
!pip install matplotlib
!pip install graphviz

# 2. MACHINE LEARNING
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 3. SQL INTEGRATION (Run '!pip install pandasql' in a separate cell if not installed)
!pip install pandasql
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())


Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [2]:
# Setting my Kaggle API token for kagglehub authentication
os.environ["KAGGLE_API_TOKEN"] = "KGAT_730ae8d7982c97f35d20f047cc217e10"

import kagglehub
path = kagglehub.competition_download("home-credit-default-risk")
print("Dataset downloaded to:", path)


Dataset downloaded to: /root/.cache/kagglehub/competitions/home-credit-default-risk


In [3]:
# Install the updated Kaggle utility if not present
!pip install kagglehub -q
import kagglehub

# Download the specific dataset requested in your proposal
path = kagglehub.competition_download("home-credit-default-risk")
print("Dataset downloaded to:", path)

Dataset downloaded to: /root/.cache/kagglehub/competitions/home-credit-default-risk


In [4]:
# Initializing SQLite database
db_path = "home_credit_data.db"
conn = sqlite3.connect(db_path)

def csv_to_sqlite(csv_name, table_name, chunksize=100000):
    """
    Streams large CSVs into SQLite in chunks to respect Colab RAM limits.
    Downcasts numeric types to save memory.
    """
    csv_file_path = os.path.join(path, csv_name)
    print(f"Ingesting {csv_name} into table '{table_name}'...")

    for i, chunk in enumerate(pd.read_csv(csv_file_path, chunksize=chunksize)):
        # Downcast to save RAM
        for col in chunk.select_dtypes(include=['float64']).columns:
            chunk[col] = chunk[col].astype('float32')
        for col in chunk.select_dtypes(include=['int64']).columns:
            chunk[col] = chunk[col].astype('int32')

        if i == 0:
            chunk.to_sql(table_name, conn, if_exists='replace', index=False)
        else:
            chunk.to_sql(table_name, conn, if_exists='append', index=False)

    print(f"  Done: '{table_name}' loaded.")

# Load all 7 tables
csv_to_sqlite("application_train.csv",    "application_train")
csv_to_sqlite("installments_payments.csv","installments_payments")
csv_to_sqlite("bureau.csv",               "bureau")
csv_to_sqlite("bureau_balance.csv",       "bureau_balance")
csv_to_sqlite("previous_application.csv", "previous_application")
csv_to_sqlite("POS_CASH_balance.csv",     "POS_CASH_balance")
csv_to_sqlite("credit_card_balance.csv",  "credit_card_balance")

print("\nAll 7 tables loaded into SQLite.")

Ingesting application_train.csv into table 'application_train'...
  Done: 'application_train' loaded.
Ingesting installments_payments.csv into table 'installments_payments'...
  Done: 'installments_payments' loaded.
Ingesting bureau.csv into table 'bureau'...
  Done: 'bureau' loaded.
Ingesting bureau_balance.csv into table 'bureau_balance'...
  Done: 'bureau_balance' loaded.
Ingesting previous_application.csv into table 'previous_application'...
  Done: 'previous_application' loaded.
Ingesting POS_CASH_balance.csv into table 'POS_CASH_balance'...
  Done: 'POS_CASH_balance' loaded.
Ingesting credit_card_balance.csv into table 'credit_card_balance'...
  Done: 'credit_card_balance' loaded.

All 7 tables loaded into SQLite.


In [5]:
# Aggregating installments_payment to main table
# FIX: Removed the wrong AND i.DAYS_INSTALMENT < 0 filter.
# DAYS_INSTALMENT and DAYS_ENTRY_PAYMENT are both negative (days before application).
# A late payment = DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT (paid later than due).
# We keep all records and filter only to before application date via the join logic.

inst_query = """
SELECT
    SK_ID_CURR,

    -- How many historical loan installments exist for this applicant
    COUNT(*)                                                        AS INST_TOTAL_RECORDS,

    -- Average delay in days (positive = late, negative = early)
    AVG(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)                       AS INST_AVG_PAYMENT_DELAY,

    -- Worst single late payment (most days past due)
    MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)                       AS INST_MAX_DAYS_PAST_DUE,

    -- Count of payments made after due date
    SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END)
                                                                    AS INST_COUNT_LATE,

    -- Proportion of payments that were late
    AVG(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1.0 ELSE 0.0 END)
                                                                    AS INST_RATIO_LATE,

    -- Average underpayment amount (positive = paid less than owed)
    AVG(AMT_INSTALMENT - AMT_PAYMENT)                               AS INST_AVG_UNDERPAYMENT

FROM installments_payments
GROUP BY SK_ID_CURR
"""

print("Aggregating installments_payments...")
df_inst = pd.read_sql_query(inst_query, conn)
print(f"  Shape: {df_inst.shape}")

Aggregating installments_payments...
  Shape: (339587, 7)


In [6]:
# Aggregate bureau
bureau_query = """
SELECT
    SK_ID_CURR,

    COUNT(SK_ID_BUREAU)                                             AS BUR_TOTAL_LOANS,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)      AS BUR_ACTIVE_LOANS,
    SUM(AMT_CREDIT_SUM_DEBT)                                        AS BUR_TOTAL_DEBT,
    MAX(AMT_CREDIT_MAX_OVERDUE)                                     AS BUR_MAX_OVERDUE_AMT,
    AVG(CREDIT_DAY_OVERDUE)                                         AS BUR_AVG_DAYS_OVERDUE,

    -- Count of loans that were prolonged (restructured) - risk signal
    SUM(CNT_CREDIT_PROLONG)                                         AS BUR_TOTAL_PROLONGED,

    -- Credit utilization: debt as share of total credit limit
    AVG(CASE
        WHEN AMT_CREDIT_SUM > 0
        THEN AMT_CREDIT_SUM_DEBT / AMT_CREDIT_SUM
        ELSE NULL
    END)                                                            AS BUR_AVG_UTILIZATION

FROM bureau
GROUP BY SK_ID_CURR
"""

print("Aggregating bureau...")
df_bur = pd.read_sql_query(bureau_query, conn)
print(f"  Shape: {df_bur.shape}")

Aggregating bureau...
  Shape: (305811, 8)


In [7]:
# Aggregate bureau_balance
# STATUS codes: 0=no DPD, 1-5=months overdue, C=closed, X=unknown
#I am going to count months in bad status as a delinquency signal

bur_bal_query = """
SELECT
    b.SK_ID_CURR,

    -- Count of months where account was in collection or overdue (status 1-5)
    SUM(CASE WHEN bb.STATUS IN ('1','2','3','4','5') THEN 1 ELSE 0 END)
                                                                    AS BURBAL_MONTHS_OVERDUE,

    -- Count of months in closed status
    SUM(CASE WHEN bb.STATUS = 'C' THEN 1 ELSE 0 END)               AS BURBAL_MONTHS_CLOSED,

    -- Total months on record (depth of credit history)
    COUNT(*)                                                        AS BURBAL_TOTAL_MONTHS

FROM bureau_balance bb
JOIN bureau b ON bb.SK_ID_BUREAU = b.SK_ID_BUREAU
GROUP BY b.SK_ID_CURR
"""

print("Aggregating bureau_balance...")
df_bur_bal = pd.read_sql_query(bur_bal_query, conn)
print(f"  Shape: {df_bur_bal.shape}")

Aggregating bureau_balance...
  Shape: (134542, 4)


In [8]:
# Aggregate previous_application
prev_query = """
SELECT
    SK_ID_CURR,

    COUNT(SK_ID_PREV)                                               AS PREV_TOTAL_APPS,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END)
                                                                    AS PREV_COUNT_APPROVED,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END)
                                                                    AS PREV_COUNT_REFUSED,

    -- Approval rate from prior applications
    AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1.0 ELSE 0.0 END)
                                                                    AS PREV_APPROVAL_RATE,

    AVG(AMT_CREDIT)                                                 AS PREV_AVG_CREDIT_AMT,
    AVG(AMT_ANNUITY)                                                AS PREV_AVG_ANNUITY

FROM previous_application
GROUP BY SK_ID_CURR
"""

print("Aggregating previous_application...")
df_prev = pd.read_sql_query(prev_query, conn)
print(f"  Shape: {df_prev.shape}")

Aggregating previous_application...
  Shape: (338857, 7)


In [9]:
# Aggregate POS_CASH_balance
pos_query = """
SELECT
    SK_ID_CURR,

    AVG(SK_DPD)                                                     AS POS_AVG_DPD,
    MAX(SK_DPD)                                                     AS POS_MAX_DPD,

    -- Count of months with any days past due
    SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)                    AS POS_MONTHS_DPD_GT0,

    -- Completed installments ratio (progress through loan)
    AVG(CASE
        WHEN CNT_INSTALMENT > 0
        THEN (CNT_INSTALMENT - CNT_INSTALMENT_FUTURE) / CNT_INSTALMENT
        ELSE NULL
    END)                                                            AS POS_AVG_COMPLETION_RATIO

FROM POS_CASH_balance
GROUP BY SK_ID_CURR
"""

print("Aggregating POS_CASH_balance...")
df_pos = pd.read_sql_query(pos_query, conn)
print(f"  Shape: {df_pos.shape}")

Aggregating POS_CASH_balance...
  Shape: (337252, 5)


In [10]:
# Aggregate credit_card_balance
cc_query = """
SELECT
    SK_ID_CURR,

    -- Average credit utilization (balance / limit) — key risk signal
    AVG(CASE
        WHEN AMT_CREDIT_LIMIT_ACTUAL > 0
        THEN AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL
        ELSE NULL
    END)                                                            AS CC_AVG_UTILIZATION,

    MAX(SK_DPD)                                                     AS CC_MAX_DPD,

    -- Average payment ratio (how much of minimum do they pay)
    AVG(CASE
        WHEN AMT_INST_MIN_REGULARITY > 0
        THEN AMT_PAYMENT_CURRENT / AMT_INST_MIN_REGULARITY
        ELSE NULL
    END)                                                            AS CC_AVG_PAYMENT_RATIO,

    COUNT(DISTINCT SK_ID_PREV)                                      AS CC_TOTAL_CARDS

FROM credit_card_balance
GROUP BY SK_ID_CURR
"""

print("Aggregating credit_card_balance...")
df_cc = pd.read_sql_query(cc_query, conn)
print(f"  Shape: {df_cc.shape}")

Aggregating credit_card_balance...
  Shape: (103558, 5)


In [11]:
# Pull selected columns from application_train
# DECISION LOG:
# DROPPED — Building/housing metrics (~40 cols): >60% missing, irrelevant to loan behavior
# DROPPED — FLAG_DOCUMENT_2 to FLAG_DOCUMENT_21: near-zero variance, no predictive value
# DROPPED — Contact flags (FLAG_MOBIL etc.): operational metadata, not behavioral
# DROPPED — Address discrepancy flags: minimal signal for default
# DROPPED — WEEKDAY/HOUR_APPR_PROCESS_START: operational, not predictive
# DROPPED — NAME_TYPE_SUITE: who accompanied applicant, irrelevant
# DROPPED — OWN_CAR_AGE: 66% missing, FLAG_OWN_CAR captures same signal
# KEPT for fairness audit only (not model input): CODE_GENDER, DAYS_BIRTH

app_query = """
SELECT
    SK_ID_CURR,
    TARGET,

    -- Fairness audit only (excluded from Model B features)
    CODE_GENDER,
    DAYS_BIRTH,

    -- Core financials
    AMT_INCOME_TOTAL,
    AMT_CREDIT,
    AMT_ANNUITY,
    AMT_GOODS_PRICE,
    NAME_CONTRACT_TYPE,

    -- Employment & stability
    DAYS_EMPLOYED,
    DAYS_REGISTRATION,
    DAYS_ID_PUBLISH,
    OCCUPATION_TYPE,
    ORGANIZATION_TYPE,
    NAME_INCOME_TYPE,

    -- Demographics
    NAME_EDUCATION_TYPE,
    NAME_FAMILY_STATUS,
    NAME_HOUSING_TYPE,
    CNT_CHILDREN,
    CNT_FAM_MEMBERS,
    FLAG_OWN_CAR,
    FLAG_OWN_REALTY,

    -- External credit scores (top predictors — keep despite missingness)
    EXT_SOURCE_1,
    EXT_SOURCE_2,
    EXT_SOURCE_3,

    -- Regional risk (ethics flagged: may proxy for ethnicity)
    REGION_RATING_CLIENT,

    -- Social circle defaults (behavioral signal)
    DEF_30_CNT_SOCIAL_CIRCLE,
    DEF_60_CNT_SOCIAL_CIRCLE,

    -- Bureau inquiry velocity
    AMT_REQ_CREDIT_BUREAU_YEAR,
    AMT_REQ_CREDIT_BUREAU_QRT,
    AMT_REQ_CREDIT_BUREAU_MON

FROM application_train
"""

print("Pulling selected application_train columns...")
df_app = pd.read_sql_query(app_query, conn)
conn.close()
print(f"  Shape: {df_app.shape}")



Pulling selected application_train columns...
  Shape: (307511, 31)


In [12]:
# Merge all aggregated tables into one master matrix
print("Merging all tables...")

df_master = df_app.copy()
df_master = df_master.merge(df_inst,    on='SK_ID_CURR', how='left')
df_master = df_master.merge(df_bur,     on='SK_ID_CURR', how='left')
df_master = df_master.merge(df_bur_bal, on='SK_ID_CURR', how='left')
df_master = df_master.merge(df_prev,    on='SK_ID_CURR', how='left')
df_master = df_master.merge(df_pos,     on='SK_ID_CURR', how='left')
df_master = df_master.merge(df_cc,      on='SK_ID_CURR', how='left')

print(f"Master matrix shape: {df_master.shape}")
print(f"Total features: {df_master.shape[1] - 2} (excluding SK_ID_CURR and TARGET)")


Merging all tables...
Master matrix shape: (307511, 61)
Total features: 59 (excluding SK_ID_CURR and TARGET)


In [13]:

# Feature engineering — composite ratios ---
print("Engineering composite features...")

# Fix the 365243 anomaly (encodes retired/unemployed, not 1000 years employed)
df_master['DAYS_EMPLOYED'] = df_master['DAYS_EMPLOYED'].replace(365243, np.nan)

# Core financial stress ratios
df_master['RATIO_DEBT_TO_INCOME']    = df_master['AMT_CREDIT']  / df_master['AMT_INCOME_TOTAL'].replace(0, np.nan)
df_master['RATIO_ANNUITY_TO_INCOME'] = df_master['AMT_ANNUITY'] / df_master['AMT_INCOME_TOTAL'].replace(0, np.nan)
df_master['RATIO_CREDIT_TO_GOODS']   = df_master['AMT_CREDIT']  / df_master['AMT_GOODS_PRICE'].replace(0, np.nan)

# Age in years (DAYS_BIRTH is negative)
df_master['AGE_YEARS'] = df_master['DAYS_BIRTH'].abs() / 365

# Employment length in years
df_master['EMPLOYED_YEARS'] = df_master['DAYS_EMPLOYED'].abs() / 365

print(f"  Engineered features added. Final shape: {df_master.shape}")


Engineering composite features...
  Engineered features added. Final shape: (307511, 66)


In [14]:

# Missing value report
print("=== MISSING VALUE AUDIT ===")

missing = pd.DataFrame({
    'Missing Count': df_master.isnull().sum(),
    'Missing %':     (df_master.isnull().sum() / len(df_master) * 100).round(2)
}).sort_values('Missing %', ascending=False)

print(missing[missing['Missing %'] > 0].to_string())
print(f"\nTotal columns: {df_master.shape[1]}")
print(f"Columns with >50% missing: {(missing['Missing %'] > 50).sum()}")
print(f"\nNOTE: High missingness in behavioral aggregates (INST_, BUR_, CC_) represents")
print(f"thin-file applicants with no prior loan history — this is expected and informative.")


=== MISSING VALUE AUDIT ===
                            Missing Count  Missing %
CC_AVG_PAYMENT_RATIO               248242      80.73
CC_AVG_UTILIZATION                 221475      72.02
CC_MAX_DPD                         220606      71.74
CC_TOTAL_CARDS                     220606      71.74
BURBAL_TOTAL_MONTHS                215280      70.01
BURBAL_MONTHS_CLOSED               215280      70.01
BURBAL_MONTHS_OVERDUE              215280      70.01
EXT_SOURCE_1                       173378      56.38
BUR_MAX_OVERDUE_AMT                123625      40.20
OCCUPATION_TYPE                     96391      31.35
EXT_SOURCE_3                        60965      19.83
DAYS_EMPLOYED                       55374      18.01
EMPLOYED_YEARS                      55374      18.01
BUR_AVG_UTILIZATION                 52634      17.12
BUR_TOTAL_DEBT                      51380      16.71
BUR_ACTIVE_LOANS                    44020      14.31
BUR_AVG_DAYS_OVERDUE                44020      14.31
BUR_TOTAL_LOANS   

In [16]:

# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/CIND820_Project'
os.makedirs(save_dir, exist_ok=True)

df_master.to_csv(os.path.join(save_dir, 'master_feature_matrix.csv'), index=False)
print(f"Saved! Shape: {df_master.shape}")
print(f"Location: {save_dir}/master_feature_matrix.csv")



Mounted at /content/drive
Saved! Shape: (307511, 66)
Location: /content/drive/MyDrive/CIND820_Project/master_feature_matrix.csv


In [17]:
# Quick Double Check
print("Double Check")
print(f"\nRows: {len(df_master):,}")
print(f"Columns: {df_master.shape[1]}")
print(f"\nClass balance:")
print(df_master['TARGET'].value_counts(normalize=True).round(3))
print(f"\nSample of engineered features:")
print(df_master[['RATIO_DEBT_TO_INCOME', 'RATIO_ANNUITY_TO_INCOME',
                  'INST_AVG_PAYMENT_DELAY', 'BUR_TOTAL_LOANS',
                  'CC_AVG_UTILIZATION']].describe().round(3))

Double Check

Rows: 307,511
Columns: 66

Class balance:
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64

Sample of engineered features:
       RATIO_DEBT_TO_INCOME  RATIO_ANNUITY_TO_INCOME  INST_AVG_PAYMENT_DELAY  \
count            307511.000               307499.000              291635.000   
mean                  3.958                    0.181                 -11.202   
std                   2.690                    0.095                  13.149   
min                   0.005                    0.000                -295.000   
25%                   2.019                    0.115                 -14.831   
50%                   3.265                    0.163                  -9.542   
75%                   5.160                    0.229                  -5.855   
max                  84.737                    1.876                1884.205   

       BUR_TOTAL_LOANS  CC_AVG_UTILIZATION  
count       263491.000           86036.000  
mean             5.561               0.